# Final Model Preparation (Final Preprocessing)

In [2]:
import os
import numpy as np
import pandas as pd

from IPython.display import display

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Define paths

In [3]:
BASE_DIR = "/content/drive/MyDrive/ML Assignment"

FEATURE_DIR = os.path.join(
    BASE_DIR,
    "engineered_dataset"
)

FINAL_MODEL_DIR = os.path.join(
    BASE_DIR,
    "final_dataset"
)

os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

print("Feature directory:")
print(FEATURE_DIR)

print("\nFinal model directory:")
print(FINAL_MODEL_DIR)

print("\nAvailable feature-engineered files:")
print(os.listdir(FEATURE_DIR))

Feature directory:
/content/drive/MyDrive/ML Assignment/engineered_dataset

Final model directory:
/content/drive/MyDrive/ML Assignment/final_dataset

Available feature-engineered files:
['X_train_features.csv', 'X_validation_features.csv', 'X_test_features.csv', 'y_train_features.csv', 'y_validation_features.csv', 'y_test_features.csv', 'feature_dictionary.csv']


Load feature engineered dataset

In [4]:
X_train = pd.read_csv(
    os.path.join(FEATURE_DIR, "X_train_features.csv")
)

X_val = pd.read_csv(
    os.path.join(FEATURE_DIR, "X_validation_features.csv")
)

X_test = pd.read_csv(
    os.path.join(FEATURE_DIR, "X_test_features.csv")
)

y_train = pd.read_csv(
    os.path.join(FEATURE_DIR, "y_train_features.csv")
).squeeze("columns")

y_val = pd.read_csv(
    os.path.join(FEATURE_DIR, "y_validation_features.csv")
).squeeze("columns")

y_test = pd.read_csv(
    os.path.join(FEATURE_DIR, "y_test_features.csv")
).squeeze("columns")

FE_OUTPUT_FEATURES = X_train.shape[1]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (18183, 159) (18183,)
Validation: (3873, 159) (3873,)
Test: (3923, 159) (3923,)


Verify alignment

In [5]:
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("Row and column alignment verified.")

Row and column alignment verified.


Target Encoding

In [6]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_val_encoded = target_encoder.transform(y_val)
y_test_encoded = target_encoder.transform(y_test)

print("Target class mapping:")
for idx, label in enumerate(target_encoder.classes_):
    print(f"  {label} -> {idx}")

Target class mapping:
  Away Win -> 0
  Draw -> 1
  Home Win -> 2


In [7]:
BURN_IN_K = 5

def burn_in_mask(X, k=BURN_IN_K):
    return ((X["home_matches_before"] >= k) & (X["away_matches_before"] >= k)).values

# Burn-in: drop cold-start rows from TRAINING only. Validation and test stay complete.
tr_m = burn_in_mask(X_train)

X_train = X_train[tr_m].reset_index(drop=True)
y_train_encoded = y_train_encoded[tr_m]

print(f"Training rows removed (fewer than {BURN_IN_K} prior matches): {(~tr_m).sum()}")
print(f"Training rows kept: {len(X_train)}")
print(f"Validation/Test kept in full: {len(X_val)} / {len(X_test)}")
print(f"(cold-start rows left in validation / test: "
      f"{(~burn_in_mask(X_val)).sum()} / {(~burn_in_mask(X_test)).sum()})")

Training rows removed (fewer than 5 prior matches): 889
Training rows kept: 17294
Validation/Test kept in full: 3873 / 3923
(cold-start rows left in validation / test: 55 / 64)


Inspect feature types

In [8]:
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:")
for col in categorical_columns:
    print("-", col)

print("\nNumber of categorical columns:",
      len(categorical_columns))

print("\nNumber of numerical columns:",
      len(numerical_columns))

Categorical columns:
- season
- league_name
- country_name
- home_team
- away_team

Number of categorical columns: 5

Number of numerical columns: 154


Identify categorical and structured event columns

In [9]:
categorical_candidates = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical/object columns detected:")
for col in categorical_candidates:
    print(
        f"- {col}: "
        f"{X_train[col].nunique():,} unique values"
    )

Categorical/object columns detected:
- season: 6 unique values
- league_name: 11 unique values
- country_name: 11 unique values
- home_team: 272 unique values
- away_team: 272 unique values


Define event-data columns

In [10]:
EVENT_COLUMNS = [
    "goal",
    "shoton",
    "shotoff",
    "foulcommit",
    "card",
    "cross",
    "corner",
    "possession"
]

events_present = [col for col in EVENT_COLUMNS if col in X_train.columns]

print("Match-event columns found in the loaded data (expected: none):")
for col in events_present:
    print("-", col)

Match-event columns found in the loaded data (expected: none):


Define other columns to exclude

In [11]:
TEAM_IDENTITY_COLUMNS = [
    "home_team",
    "away_team"
]

TECHNICAL_COLUMNS = [
    "id",
    "match_id",
    "country_id",
    "league_id",
    "home_team_api_id",
    "away_team_api_id",
    "date",
    "country_name",
    "season"
]

PLAYER_ID_COLUMNS = [
    f"home_player_{i}" for i in range(1, 12)
] + [
    f"away_player_{i}" for i in range(1, 12)
]

TARGET_LEAKAGE_COLUMNS = [
    "home_team_goal",
    "away_team_goal",
    "target_outcome",
    "total_goals",
    "goal_difference"
]

In [12]:
# BOOKMAKER IMPLIED PROBABILITY FEATURES
BOOKMAKER_TRIPLETS = {
    "B365": ("B365H", "B365D", "B365A"),
    "BW":   ("BWH",   "BWD",   "BWA"),
    "IW":   ("IWH",   "IWD",   "IWA"),
    "LB":   ("LBH",   "LBD",   "LBA"),
    "PS":   ("PSH",   "PSD",   "PSA"),
    "WH":   ("WHH",   "WHD",   "WHA"),
    "SJ":   ("SJH",   "SJD",   "SJA"),
    "VC":   ("VCH",   "VCD",   "VCA"),
    "GB":   ("GBH",   "GBD",   "GBA"),
    "BS":   ("BSH",   "BSD",   "BSA"),
}
ODDS_COLS = [c for t in BOOKMAKER_TRIPLETS.values() for c in t]

# Only bookmakers that are available in ALL splits (checked in the availability table in the next cell)
STABLE_BOOKS = ["B365", "BW", "IW", "LB", "WH", "VC"]

def add_market_probability_features(df):
    df = df.copy()

    home_probability_columns = []
    draw_probability_columns = []
    away_probability_columns = []

    for bookmaker, columns in BOOKMAKER_TRIPLETS.items():

        if bookmaker not in STABLE_BOOKS:
            continue

        home_col, draw_col, away_col = columns

        # Skip bookmaker if these columns are unavailable
        if not all(
            col in df.columns
            for col in columns
        ):
            continue

        # Protect against impossible/non-positive odds
        valid = (
            (df[home_col] > 0)
            & (df[draw_col] > 0)
            & (df[away_col] > 0)
        )

        raw_home = np.where(valid, 1.0 / df[home_col], np.nan)
        raw_draw = np.where(valid, 1.0 / df[draw_col], np.nan)
        raw_away = np.where(valid, 1.0 / df[away_col], np.nan)

        total = (raw_home + raw_draw + raw_away)

        home_name = (f"{bookmaker}_implied_home_probability")
        draw_name = (f"{bookmaker}_implied_draw_probability")
        away_name = (f"{bookmaker}_implied_away_probability")

        df[home_name] = raw_home / total
        df[draw_name] = raw_draw / total
        df[away_name] = raw_away / total

        home_probability_columns.append(home_name)
        draw_probability_columns.append(draw_name)
        away_probability_columns.append(away_name)

    # Consensus market probabilities
    if home_probability_columns:

        df["market_home_probability"] = (
            df[home_probability_columns].mean(axis=1)
        )

        df["market_draw_probability"] = (
            df[draw_probability_columns].mean(axis=1)
        )

        df["market_away_probability"] = (
            df[away_probability_columns].mean(axis=1)
        )

        # Bookmaker disagreement
        df["market_home_probability_std"] = (
            df[home_probability_columns].std(axis=1)
        )

        df["market_draw_probability_std"] = (
            df[draw_probability_columns].std(axis=1)
        )

        df["market_away_probability_std"] = (
            df[away_probability_columns].std(axis=1)
        )

    return df

X_train = add_market_probability_features(X_train)
X_val = add_market_probability_features(X_val)
X_test = add_market_probability_features(X_test)

MARKET_FEATURES = [
    "market_home_probability",
    "market_draw_probability",
    "market_away_probability",
    "market_home_probability_std",
    "market_draw_probability_std",
    "market_away_probability_std"
]

MARKET_FEATURES = [
    col
    for col in MARKET_FEATURES
    if col in X_train.columns
]

print("Consensus market features created:")

for col in MARKET_FEATURES:
    print("-", col)

# Add market missing flag and fill consensus features with train medians
for df in (X_train, X_val, X_test):
    df["market_missing"] = df["market_home_probability"].isna().astype(int)

# Compute fill values strictly from the training set
mkt_fill = X_train[MARKET_FEATURES].median()

# Apply fill to training, validation, and test sets
for df in (X_train, X_val, X_test):
    df[MARKET_FEATURES] = df[MARKET_FEATURES].fillna(mkt_fill)

print("Market missing flags added and consensus features imputed from train medians.")

Consensus market features created:
- market_home_probability
- market_draw_probability
- market_away_probability
- market_home_probability_std
- market_draw_probability_std
- market_away_probability_std
Market missing flags added and consensus features imputed from train medians.


In [13]:
# Odds availability per bookmaker and split (home-odds column used as proxy) - mirrors DP-4
book_missing = pd.DataFrame({
    book: {"train %": X_train[h].isna().mean() * 100,
           "val %":   X_val[h].isna().mean() * 100,
           "test %":  X_test[h].isna().mean() * 100}
    for book, (h, _, _) in BOOKMAKER_TRIPLETS.items()
}).T.round(1)
display(book_missing)

unstable = book_missing[(book_missing > 50).any(axis=1)].index.tolist()
print("Unstable bookmakers (>50% missing in at least one split):", unstable)

bad = sorted(set(STABLE_BOOKS) & set(unstable))
if bad:
    raise ValueError(f"STABLE_BOOKS contains bookmakers that are unstable across splits: {bad}")

stable_odds = [c for b in STABLE_BOOKS for c in BOOKMAKER_TRIPLETS[b]]
for name, d in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(f"{name}: all-stable-odds-missing rate =",
          d[stable_odds].isna().all(axis=1).mean().round(3))

,train %,val %,test %
B365,13.1,13.1,12.8
BW,13.2,13.1,12.8
IW,13.4,13.6,13.1
LB,13.3,13.3,12.8
PS,75.3,13.5,13.0
WH,13.2,13.1,12.8
SJ,13.2,66.2,100.0
VC,13.2,13.1,12.8
GB,22.3,100.0,100.0
BS,22.3,100.0,100.0


Unstable bookmakers (>50% missing in at least one split): ['PS', 'SJ', 'GB', 'BS']
train: all-stable-odds-missing rate = 0.131
val: all-stable-odds-missing rate = 0.131
test: all-stable-odds-missing rate = 0.127


In [14]:
# Define lineup coordinate columns (44 columns)
XY_COLS = [f"{s}_player_{a}{i}" for s in ("home","away") for a in ("X","Y") for i in range(1,12)]

# Identify per-bookmaker implied probability columns dynamically
PER_BOOK_PROBS = [c for c in X_train.columns if "_implied_" in c]

COUNTER_COLS = [
    "home_matches_before",
    "away_matches_before",
    "home_home_matches_before",
    "away_away_matches_before"
]

# Combine all exclusions into DROP_COLUMNS
DROP_COLUMNS = (
    TEAM_IDENTITY_COLUMNS
    + TECHNICAL_COLUMNS
    + PLAYER_ID_COLUMNS
    + TARGET_LEAKAGE_COLUMNS
    + EVENT_COLUMNS
    + ODDS_COLS
    + XY_COLS
    + PER_BOOK_PROBS
    + [c for c in COUNTER_COLS if c in X_train.columns]
)

# Remove duplicates while preserving order
DROP_COLUMNS = list(
    dict.fromkeys(
        col
        for col in DROP_COLUMNS
        if col in X_train.columns
    )
)

print("Columns excluded from final modelling:")
for col in DROP_COLUMNS:
    print("-", col)

Columns excluded from final modelling:
- home_team
- away_team
- country_name
- season
- B365H
- B365D
- B365A
- BWH
- BWD
- BWA
- IWH
- IWD
- IWA
- LBH
- LBD
- LBA
- PSH
- PSD
- PSA
- WHH
- WHD
- WHA
- SJH
- SJD
- SJA
- VCH
- VCD
- VCA
- GBH
- GBD
- GBA
- BSH
- BSD
- BSA
- home_player_X1
- home_player_X2
- home_player_X3
- home_player_X4
- home_player_X5
- home_player_X6
- home_player_X7
- home_player_X8
- home_player_X9
- home_player_X10
- home_player_X11
- home_player_Y1
- home_player_Y2
- home_player_Y3
- home_player_Y4
- home_player_Y5
- home_player_Y6
- home_player_Y7
- home_player_Y8
- home_player_Y9
- home_player_Y10
- home_player_Y11
- away_player_X1
- away_player_X2
- away_player_X3
- away_player_X4
- away_player_X5
- away_player_X6
- away_player_X7
- away_player_X8
- away_player_X9
- away_player_X10
- away_player_X11
- away_player_Y1
- away_player_Y2
- away_player_Y3
- away_player_Y4
- away_player_Y5
- away_player_Y6
- away_player_Y7
- away_player_Y8
- away_player_Y9
- away_

Remove raw team identity + technical fields

In [15]:
X_train_base = X_train.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
).copy()

X_val_base = X_val.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
).copy()

X_test_base = X_test.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
).copy()

print("Training shape:", X_train_base.shape)
print("Validation shape:", X_val_base.shape)
print("Test shape:", X_test_base.shape)

Training shape: (17294, 84)
Validation shape: (3873, 84)
Test shape: (3923, 84)


Re-identify categorical variables

In [16]:
categorical_features = X_train_base.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train_base.select_dtypes(
    include=np.number
).columns.tolist()

print("Remaining categorical variables:")
for col in categorical_features:
    print(
        f"- {col}: "
        f"{X_train_base[col].nunique():,} unique values"
    )

print("\nNumber of numerical features:",
      len(numerical_features))

Remaining categorical variables:
- league_name: 11 unique values

Number of numerical features: 83


Verify no event columns remain

In [17]:
remaining_event_columns = [
    col
    for col in EVENT_COLUMNS
    if col in X_train_base.columns
]

print(
    "Event columns remaining:",
    remaining_event_columns
)

assert len(remaining_event_columns) == 0

print("Event-column removal verified.")

Event columns remaining: []
Event-column removal verified.


Check categorical cardinality

In [18]:
categorical_cardinality = pd.DataFrame({
    "Column": categorical_features,
    "Unique Values": [
        X_train_base[col].nunique()
        for col in categorical_features
    ]
}).sort_values(
    "Unique Values",
    ascending=False
)

display(categorical_cardinality)

,Column,Unique Values
0,league_name,11


One-hot encode only the genuine categorical variables

In [19]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

In [20]:
# Fit only on training
X_train_cat = encoder.fit_transform(
    X_train_base[categorical_features]
)

X_val_cat = encoder.transform(
    X_val_base[categorical_features]
)

X_test_cat = encoder.transform(
    X_test_base[categorical_features]
)

Inspect encoded dimensionality

In [21]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

print(
    "Original categorical columns:",
    len(categorical_features)
)

print(
    "Encoded categorical features:",
    len(encoded_columns)
)

print("\nEncoded columns:")
for col in encoded_columns:
    print("-", col)

Original categorical columns: 1
Encoded categorical features: 10

Encoded columns:
- league_name_England Premier League
- league_name_France Ligue 1
- league_name_Germany 1. Bundesliga
- league_name_Italy Serie A
- league_name_Netherlands Eredivisie
- league_name_Poland Ekstraklasa
- league_name_Portugal Liga ZON Sagres
- league_name_Scotland Premier League
- league_name_Spain LIGA BBVA
- league_name_Switzerland Super League


Convert encoded categories to DataFrames

In [22]:
X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_columns
)

X_val_cat = pd.DataFrame(
    X_val_cat,
    columns=encoded_columns
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_columns
)

display(X_train_cat.head())

,league_name_England Premier League,league_name_France Ligue 1,league_name_Germany 1. Bundesliga,league_name_Italy Serie A,league_name_Netherlands Eredivisie,league_name_Poland Ekstraklasa,league_name_Portugal Liga ZON Sagres,league_name_Scotland Premier League,league_name_Spain LIGA BBVA,league_name_Switzerland Super League
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Numerical features

In [23]:
X_train_num_unscaled = (
    X_train_base[numerical_features]
    .reset_index(drop=True)
)

X_val_num_unscaled = (
    X_val_base[numerical_features]
    .reset_index(drop=True)
)

X_test_num_unscaled = (
    X_test_base[numerical_features]
    .reset_index(drop=True)
)

Feature Scaling

In [24]:
# SCALE NUMERICAL FEATURES FOR LOGISTIC REGRESSION ONLY
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num_unscaled),
    columns=numerical_features
)

X_val_num_scaled = pd.DataFrame(
    scaler.transform(X_val_num_unscaled),
    columns=numerical_features
)

X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num_unscaled),
    columns=numerical_features
)

print("Scaled numerical representation created for Logistic Regression.")

Scaled numerical representation created for Logistic Regression.


Combine numerical & encoded categorical features

In [25]:
# LOGISTIC REGRESSION MODEL MATRIX
# Scaled numerical + encoded categorical

X_train_logistic = pd.concat(
    [
        X_train_num_scaled.reset_index(drop=True),
        X_train_cat.reset_index(drop=True)
    ],
    axis=1
)

X_val_logistic = pd.concat(
    [
        X_val_num_scaled.reset_index(drop=True),
        X_val_cat.reset_index(drop=True)
    ],
    axis=1
)

X_test_logistic = pd.concat(
    [
        X_test_num_scaled.reset_index(drop=True),
        X_test_cat.reset_index(drop=True)
    ],
    axis=1
)

print("Logistic Regression matrices:")
print("Training  :", X_train_logistic.shape)
print("Validation:", X_val_logistic.shape)
print("Test      :", X_test_logistic.shape)

Logistic Regression matrices:
Training  : (17294, 93)
Validation: (3873, 93)
Test      : (3923, 93)


In [26]:
# TREE MODEL MATRIX - Decision Tree / Random Forest / XGBoost
# Unscaled numerical + encoded categorical

X_train_tree = pd.concat(
    [
        X_train_num_unscaled.reset_index(drop=True),
        X_train_cat.reset_index(drop=True)
    ],
    axis=1
)

X_val_tree = pd.concat(
    [
        X_val_num_unscaled.reset_index(drop=True),
        X_val_cat.reset_index(drop=True)
    ],
    axis=1
)

X_test_tree = pd.concat(
    [
        X_test_num_unscaled.reset_index(drop=True),
        X_test_cat.reset_index(drop=True)
    ],
    axis=1
)

print("Tree-model matrices:")
print("Training  :", X_train_tree.shape)
print("Validation:", X_val_tree.shape)
print("Test      :", X_test_tree.shape)

Tree-model matrices:
Training  : (17294, 93)
Validation: (3873, 93)
Test      : (3923, 93)


Verify same feature space

In [27]:
# Logistic Regression consistency
assert list(X_train_logistic.columns) == list(
    X_val_logistic.columns
)

assert list(X_train_logistic.columns) == list(
    X_test_logistic.columns
)

# Tree-model consistency
assert list(X_train_tree.columns) == list(
    X_val_tree.columns
)

assert list(X_train_tree.columns) == list(
    X_test_tree.columns
)

# Both representations should contain the same feature names
assert list(X_train_logistic.columns) == list(
    X_train_tree.columns
)

print("Feature-space consistency verified.")

Feature-space consistency verified.


Show the dimensionality reduction

In [28]:
# SHOW FEATURE-SPACE TRANSFORMATION
dimension_summary = pd.DataFrame({
    "Stage": [
        "Feature-engineered output (loaded)",
        "After adding market-probability features",
        "After removing event/team/technical/odds columns",
        "Logistic Regression model-ready representation",
        "Tree models model-ready representation"
    ],

    "Features": [
        FE_OUTPUT_FEATURES,
        X_train.shape[1],
        X_train_base.shape[1],
        X_train_logistic.shape[1],
        X_train_tree.shape[1]
    ]
})

display(dimension_summary)

,Stage,Features
0,Feature-engineered output (loaded),159
1,After adding market-probability features,184
2,After removing event/team/technical/odds columns,84
3,Logistic Regression model-ready representation,93
4,Tree models model-ready representation,93


In [29]:
print(f"Feature-engineered output (loaded)     : {FE_OUTPUT_FEATURES:,}")
print(f"After adding market-probability features: {X_train.shape[1]:,}")
print(f"After exclusion                          : {X_train_base.shape[1]:,}")
print(f"Logistic Regression features             : {X_train_logistic.shape[1]:,}")
print(f"Tree-model features                      : {X_train_tree.shape[1]:,}")

Feature-engineered output (loaded)     : 159
After adding market-probability features: 184
After exclusion                          : 84
Logistic Regression features             : 93
Tree-model features                      : 93


Final leakage check

In [30]:
LEAKAGE_COLUMNS = (
    TARGET_LEAKAGE_COLUMNS
    + EVENT_COLUMNS
)

# Identifiers / raw fields that must never reach a model
FORBIDDEN_COLUMNS = (
    LEAKAGE_COLUMNS
    + TECHNICAL_COLUMNS
    + TEAM_IDENTITY_COLUMNS
    + PLAYER_ID_COLUMNS
    + ODDS_COLS
)

for dataset_name, df in [
    ("Logistic", X_train_logistic),
    ("Tree", X_train_tree)
]:

    remaining = [
        col
        for col in FORBIDDEN_COLUMNS
        if col in df.columns
    ]

    print(
        dataset_name,
        "leakage/identifier/raw-odds columns:",
        remaining
    )

    assert len(remaining) == 0

print("Final leakage checks passed.")

Logistic leakage/identifier/raw-odds columns: []
Tree leakage/identifier/raw-odds columns: []
Final leakage checks passed.


Final missing/infinite check

In [31]:
for name, df in [
    ("Logistic Training", X_train_logistic),
    ("Logistic Validation", X_val_logistic),
    ("Logistic Test", X_test_logistic),

    ("Tree Training", X_train_tree),
    ("Tree Validation", X_val_tree),
    ("Tree Test", X_test_tree)
]:

    missing = df.isnull().sum().sum()

    infinite = np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()

    print(
        f"{name}: "
        f"missing={missing}, "
        f"infinite={infinite}"
    )

    assert missing == 0
    assert infinite == 0

print("Missing/infinite-value checks passed.")

Logistic Training: missing=0, infinite=0
Logistic Validation: missing=0, infinite=0
Logistic Test: missing=0, infinite=0
Tree Training: missing=0, infinite=0
Tree Validation: missing=0, infinite=0
Tree Test: missing=0, infinite=0
Missing/infinite-value checks passed.


Verify alignments before saving to prevent silent errors

In [32]:
assert len(X_train_logistic) == len(X_train_tree) == len(y_train_encoded)
assert len(X_val_logistic) == len(X_val_tree) == len(y_val_encoded)
assert len(X_test_logistic) == len(X_test_tree) == len(y_test_encoded)

# Validation and test must be the COMPLETE chronological splits (burn-in applies to training only)
assert len(X_val_logistic) == len(y_val), "Validation set was shrunk"
assert len(X_test_logistic) == len(y_test), "Test set was shrunk"

print("Dataset alignment assertions passed successfully.")
print(f"Rows -> train: {len(X_train_tree)}, validation: {len(X_val_tree)}, test: {len(X_test_tree)}")

Dataset alignment assertions passed successfully.
Rows -> train: 17294, validation: 3873, test: 3923


In [33]:
for name, d in [("train", X_train_tree), ("validation", X_val_tree), ("test", X_test_tree)]:
    print(f"market_missing rate ({name}):", d["market_missing"].mean().round(3))

odds_left = [c for c in X_train_tree.columns if c in ODDS_COLS or "_implied_" in c]
print("Raw odds left:", odds_left)          # must be []
assert odds_left == []

eng = set(pd.read_csv(os.path.join(FEATURE_DIR, "feature_dictionary.csv"))["Feature"])
extra = [c for c in X_train_tree.columns
         if c not in eng and not c.startswith(("market_", "league_name_"))]
print("Unexpected columns:", extra)         # should be only ['stage']

market_missing rate (train): 0.131
market_missing rate (validation): 0.131
market_missing rate (test): 0.127
Raw odds left: []
Unexpected columns: ['stage']


Save final dataset

In [34]:
# SAVE MODEL-READY DATASETS
import joblib

# Logistic Regression
X_train_logistic.to_csv(
    os.path.join(FINAL_MODEL_DIR, "X_train_logistic.csv"),
    index=False
)

X_val_logistic.to_csv(
    os.path.join(FINAL_MODEL_DIR, "X_validation_logistic.csv"),
    index=False
)

X_test_logistic.to_csv(
    os.path.join(FINAL_MODEL_DIR, "X_test_logistic.csv"),
    index=False
)


# Decision Tree / Random Forest / XGBoost
X_train_tree.to_csv(
    os.path.join(FINAL_MODEL_DIR, "X_train_tree.csv"),
    index=False
)

X_val_tree.to_csv(
    os.path.join(FINAL_MODEL_DIR, "X_validation_tree.csv"),
    index=False
)

X_test_tree.to_csv(
    os.path.join(FINAL_MODEL_DIR, "X_test_tree.csv"),
    index=False
)


# Targets
pd.Series(
    y_train_encoded,name="target_outcome").to_csv(
    os.path.join(FINAL_MODEL_DIR, "y_train.csv"),
    index=False
)

pd.Series(
    y_val_encoded, name="target_outcome").to_csv(
    os.path.join(FINAL_MODEL_DIR, "y_validation.csv"),
    index=False
)

pd.Series(
    y_test_encoded,name="target_outcome").to_csv(
    os.path.join(FINAL_MODEL_DIR, "y_test.csv"),
    index=False
)


# Preprocessing artifacts
joblib.dump(
    scaler,
    os.path.join(FINAL_MODEL_DIR, "logistic_scaler.joblib")
)

joblib.dump(
    encoder,
    os.path.join(FINAL_MODEL_DIR, "encoder.joblib")
)

joblib.dump(
    target_encoder,
    os.path.join(FINAL_MODEL_DIR, "target_encoder.joblib"
    )
)

print("All final model-ready datasets saved successfully.")

All final model-ready datasets saved successfully.
